# Network training

For our project, we constructed a Vision Transformer (ViT) instance that we named `GlowDeTR` to assess how light exposure affects Neural Network's result at classification.

In [5]:
import torch
import json
import torch.nn as nn
import numpy as np
import albumentations

from pathlib import Path
from PIL import Image, ImageDraw
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from transformers import ViTModel, ViTForImageClassification, AutoImageProcessor, TrainingArguments, Trainer
from datasets import load_dataset
# huggingface-cli download yin30lei/day_and_night_traffic_lights --repo-type dataset --local-dir ./traffic_lights

In [6]:
class GlowDeTR(nn.Module):
    def __init__(self, pretrained_name="google/vit-base-patch16-224-in21k", num_classes=10):
        super(GlowDeTR, self).__init__()
        # Load pretrained ViT backbone
        self.detr = ViTForImageClassification.from_pretrained(
            pretrained_name,
            num_labels=num_classes)    

    def forward(self, pixel_values):
        # Pass through the ViT backbone
        outputs = self.detr(pixel_values=pixel_values)
        return outputs.logits
    
tensor2PIL = transforms.ToPILImage()

transform1 = albumentations.Compose(
    [
        albumentations.Resize(480, 480),
        albumentations.HorizontalFlip(p=1.0),
        albumentations.RandomBrightnessContrast(p=1.0),
    ],
    bbox_params=albumentations.BboxParams(format="coco", label_fields=["category"]),
)

checkpoint = "facebook/detr-resnet-50-dc5"
image_processor = AutoImageProcessor.from_pretrained(checkpoint)

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


In [7]:
# Parameters
batch_size = 32
train_images_path = Path.cwd() / "traffic_lights/Day and Night Traffic Sign.v3i.coco/train"
train_annotations_path = Path.cwd() / "traffic_lights/Day and Night Traffic Sign.v3i.coco/_annotations.coco_train.json"
valid_images_path = Path.cwd() / "traffic_lights/Day and Night Traffic Sign.v3i.coco/valid"
valid_annotations_path = Path.cwd() / "traffic_lights/Day and Night Traffic Sign.v3i.coco/_annotations.coco_valid.json"
test_images_path = Path.cwd() / "traffic_lights/Day and Night Traffic Sign.v3i.coco/test"
test_annotations_path = Path.cwd() / "traffic_lights/Day and Night Traffic Sign.v3i.coco/_annotations.coco_test.json"


with open(train_annotations_path, "r") as f:
    annotations = json.load(f)


# number of classes
categories = annotations.get("categories", [])
num_classes = len(categories)

print(f"Number of classes: {num_classes}")

def collate_fn(batch):
    images, targets = zip(*batch)  # Unzip the batch
    return torch.stack(images), list(targets)

# Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Update normalization stats as needed for your dataset
])

# Load datasets
train_dataset = datasets.CocoDetection(
    root=train_images_path,
    annFile=train_annotations_path,
    transform=transform
)

val_dataset = datasets.CocoDetection(
    root=valid_images_path,
    annFile=valid_annotations_path,
    transform=transform
)

test_dataset = datasets.CocoDetection(
    root=test_images_path,
    annFile=test_annotations_path,
    transform=transform
)

# Define DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# print("Testing train_loader...")
# for images, targets in train_loader:
#     print("Images shape:", images.shape)
#     print("Targets:", targets)  # COCO targets will be a list of dictionaries per image
#     break

# print("Testing valid_loader...")
# for images, targets in valid_loader:
#     print("Images shape:", images.shape)
#     print("Targets:", targets)  # COCO targets will be a list of dictionaries per image
#     break

# print("Testing test_loader...")
# for images, targets in test_loader:
#     print("Images shape:", images.shape)
#     print("Targets:", targets)  # COCO targets will be a list of dictionaries per image
#     break


Number of classes: 2
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


In [8]:
def draw_image_from_idx(dataset, idx):
    sample = dataset[idx]
    image = sample["image"]
    annotations = sample["objects"]
    draw = ImageDraw.Draw(image)
    width, height = sample["width"], sample["height"]

    for i in range(len(annotations["id"])):
        box = annotations["bbox"][i]
        class_idx = annotations["id"][i]
        x, y, w, h = tuple(box)
        if max(box) > 1.0:
            x1, y1 = int(x), int(y)
            x2, y2 = int(x + w), int(y + h)
        else:
            x1 = int(x * width)
            y1 = int(y * height)
            x2 = int((x + w) * width)
            y2 = int((y + h) * height)
        draw.rectangle((x1, y1, x2, y2), outline="red", width=1)
        draw.text((x1, y1), annotations["category"][i], fill="white")
    return image


In [9]:
# # Hyperparameters
# # img_size = 32
# # patch_size = 4
# learning_rate = 1e-3
# num_epochs = 10

# # Model, Loss, Optimizer
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = GlowViT(num_classes=num_classes).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# # Training Loop
# for epoch in range(num_epochs):
#     model.train()
#     train_loss = 0.0
#     for images, labels in train_loader:
#         images, labels = images.to(device), labels.to(device)

#         # Forward pass
#         outputs = model(images)
#         loss = criterion(outputs, labels)

#         # Backward pass
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
        
#         train_loss += loss.item()
            
#     train_loss /= len(train_loader)

#     # Validation Phase
#     model.eval()  # Set model to evaluation mode
#     val_loss = 0.0
#     correct = 0
#     total = 0

#     with torch.no_grad():  # Disable gradient calculation
#         for images, labels in valid_loader:
#             images, labels = images.to(device), labels.to(device)

#             # Forward pass
#             outputs = model(images)
#             loss = criterion(outputs, labels)
#             val_loss += loss.item()

#             # Calculate accuracy
#             _, predicted = outputs.max(1)  # Get predicted class
#             total += labels.size(0)
#             correct += predicted.eq(labels).sum().item()

#     val_loss /= len(valid_loader)
#     val_accuracy = 100 * correct / total

#     # Print epoch statistics
#     print(f"Epoch [{epoch+1}/{num_epochs}], "
#           f"Train Loss: {train_loss:.4f}, "
#           f"Val Loss: {val_loss:.4f}, "
#           f"Val Accuracy: {val_accuracy:.2f}%")
#     # print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

In [10]:
# # Evaluate the Model
# model.eval()
# correct = 0
# total = 0
# with torch.no_grad():
#     for images, labels in test_loader:
#         images, labels = images.to(device), labels.to(device)
#         outputs = model(images)
#         _, predicted = torch.max(outputs, 1)
#         total += labels.size(0)
#         correct += (predicted == labels).sum().item()

# print(f"Test Accuracy: {100 * correct / total:.2f}%")

In [11]:
def preprocess_batch(images, targets, image_processor, tensor2PIL):
    processed_images = []
    processed_targets = []

    for idx in range(len(images)):
        # Convert tensor to PIL Image
        curr_pil = tensor2PIL(images[idx])

        # Format target for image_processor
        curr_target_list = targets[idx]
        target_dct = {"image_id": curr_target_list[0]["image_id"], "annotations": curr_target_list}

        # Process the image and target
        curr_out = image_processor(images=curr_pil, annotations=target_dct, return_tensors="pt")

        # Collect results
        processed_images.append(curr_out["pixel_values"].squeeze(0))
        processed_targets.append(curr_out["labels"])

    # Stack processed images into a single tensor
    processed_images = torch.stack(processed_images)
    
    return processed_images, processed_targets

# Loop through DataLoader
for images, targets in train_loader:
    processed_images, processed_targets = preprocess_batch(
        images, targets, image_processor, tensor2PIL
    )

    # Now `processed_images` and `processed_targets` are ready to be fed into your model
    print("Processed images shape:", processed_images.shape)
    print("Processed targets:", processed_targets)
    break

Processed images shape: torch.Size([32, 3, 800, 800])
Processed targets: [[{'size': tensor([800, 800]), 'image_id': tensor([258]), 'class_labels': tensor([1]), 'boxes': tensor([[0.3027, 0.2891, 0.0148, 0.0281]]), 'area': tensor([267.1875]), 'iscrowd': tensor([0]), 'orig_size': tensor([640, 640])}], [{'size': tensor([800, 800]), 'image_id': tensor([2712]), 'class_labels': tensor([1]), 'boxes': tensor([[0.0867, 0.6324, 0.0672, 0.0930]]), 'area': tensor([3997.6562]), 'iscrowd': tensor([0]), 'orig_size': tensor([640, 640])}], [{'size': tensor([800, 800]), 'image_id': tensor([2735]), 'class_labels': tensor([1]), 'boxes': tensor([[0.4820, 0.5137, 0.0203, 0.0398]]), 'area': tensor([517.9688]), 'iscrowd': tensor([0]), 'orig_size': tensor([640, 640])}], [{'size': tensor([800, 800]), 'image_id': tensor([1577]), 'class_labels': tensor([1]), 'boxes': tensor([[0.5230, 0.5273, 0.0242, 0.0484]]), 'area': tensor([750.7812]), 'iscrowd': tensor([0]), 'orig_size': tensor([640, 640])}], [{'size': tensor([

In [12]:
# print("Testing image processor")
for images, targets in train_loader:
    # print("Images shape:", images.shape)
    # print("Targets:", targets)
    for idx in range(batch_size):
        curr_target_list = targets[idx]
        target_dct = {"image_id": curr_target_list[0]["image_id"], "annotations": curr_target_list}
        curr_pil = tensor2PIL(images[idx,:,:,:])
        curr_out = image_processor(images=curr_pil, annotations=target_dct, return_tensors="pt")
        print(curr_out)
        
        if idx == 1:
            break
        
    break



{'pixel_values': tensor([[[[ 0.7591,  0.6392,  0.6906,  ..., -0.0458, -0.0458, -0.0458],
          [ 0.7762,  0.6563,  0.7077,  ..., -0.0458, -0.0458, -0.0458],
          [ 0.8276,  0.7077,  0.7419,  ..., -0.0458, -0.0458, -0.0458],
          ...,
          [ 1.3755,  1.2385,  1.2557,  ..., -1.8953, -1.7754, -1.9295],
          [ 1.2899,  1.2899,  1.3070,  ..., -1.8953, -1.8268, -2.0323],
          [ 1.1700,  1.2899,  1.3070,  ..., -1.8439, -1.8439, -2.1008]],

         [[ 0.3452,  0.2927,  0.3627,  ..., -1.1078, -1.1078, -1.1078],
          [ 0.3627,  0.3102,  0.3978,  ..., -1.1078, -1.1078, -1.1078],
          [ 0.4153,  0.3627,  0.4503,  ..., -1.1078, -1.1078, -1.1078],
          ...,
          [ 1.2556,  1.1155,  1.1331,  ...,  0.2577, -1.1779,  0.2402],
          [ 1.1681,  1.1681,  1.1856,  ..., -0.3901, -0.5651,  2.3761],
          [ 1.0455,  1.1681,  1.1856,  ..., -1.9132, -0.5826,  2.3060]],

         [[-0.0092, -0.0615,  0.0605,  ..., -1.2293, -1.2293, -1.2293],
          [ 0